# Build Micro Textbook

**Generated by** `splc --lang python/linalg`
**Domain library:** `linalg_graph.py`
**DODA note:** The source `.spl` file is the logical view; this notebook is the physical artifact for the `python/linalg` domain target.

## Inputs

| Parameter | Type | Description |
|-----------|------|-------------|
| `@target` | `TEXT` | — |
| `@style` | `TEXT` | — |
| `@primitive_budget` | `INT` | — |
| `@payoff_weight` | `FLOAT` | — |

In [1]:
# ── python/linalg target setup — generated by splc ──────────────────────────────
import sys, os, json, time
from pathlib import Path

# Locate linalg_graph.py (cookbook recipe directory or current dir)
_SEARCH = [
    Path(__file__).parent if "__file__" in dir() else Path("."),
    Path(os.environ.get("LINALG_GRAPH_DIR", ".")),
    Path("."),
]
for _p in _SEARCH:
    _p = _p.resolve()
    if (_p / "linalg_graph.py").exists():
        sys.path.insert(0, str(_p))
        break
import linalg_graph as dg
from linalg_graph import (
    build, acyclic, reducible, minimal, ancestors, restrict,
    productivity_order, in_graph, applications_of, new_primitives,
    first_radical_primitives, both_radical_primitives,
    concept_names, primitive_names,
    gap, learning_path,
)
from style_profiles import style_instruction, get_style_profile, available_styles

# ── SPL runtime config lookup — env var, then ~/.spl/config, then default
# (same precedence `spl3 configure` documents for "SPL runtime defaults";
# the notebook is a standalone artifact so it re-reads the dotenv-format
# file directly rather than importing the spl3 CLI) ──────────────────────────
def _spl_config(key: str, default: str) -> str:
    if key in os.environ:
        return os.environ[key]
    _cfg = Path.home() / ".spl" / "config"
    if _cfg.exists():
        for _line in _cfg.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _, _v = _line.partition("=")
                if _k.strip() == key:
                    return _v.strip().strip('"').strip("'")
    return default

# ── LLM helper (configure SPL_MODEL / SPL_LLM_TIMEOUT / SPL_OLLAMA_URL, or
# replace with your adapter) — calls Ollama's OpenAI-compatible HTTP endpoint
# directly (stream=False), the same way spl/adapters/ollama.py's OllamaAdapter
# does. Earlier this shelled out to the interactive `ollama run <model>` CLI —
# but that CLI streams a live "Thinking..." status line to stdout using raw
# ANSI cursor-control escapes (`\x1b[9D\x1b[K`, ...) to overwrite it in place,
# and `subprocess.run(capture_output=True)` captures those escapes verbatim
# into the "response", contaminating every GENERATE result, the cache, and the
# committed textbook with ~2000 stray control-character sequences. The HTTP
# API talks to the same Ollama daemon but returns clean structured JSON. ──────
def _llm_call(prompt: str, model: str | None = None) -> str:
    import urllib.request
    _m = model or os.environ.get("SPL_MODEL", "llama3.2")
    # Local models (esp. 8B+ on consumer GPUs) can take well over a minute per
    # call, and refine-loop prompts re-send the full prior section as context —
    # default generously and let SPL_LLM_TIMEOUT override per deployment.
    _timeout = int(os.environ.get("SPL_LLM_TIMEOUT", "600"))
    _url = os.environ.get("SPL_OLLAMA_URL", "http://localhost:11434") + "/v1/chat/completions"
    _payload = json.dumps({
        "model": _m,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }).encode("utf-8")
    _req = urllib.request.Request(
        _url, data=_payload, method="POST",
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(_req, timeout=_timeout) as _resp:
        _data = json.loads(_resp.read().decode("utf-8"))
    return _data["choices"][0]["message"]["content"].strip()

# ── Layer 2 content cache (write-once, content-addressed — same backend and
# semantics as spl/stdlib.py's cache_get/cache_put @spl_tools; reimplemented
# here directly against spl3.cache because this notebook is a standalone
# artifact that calls the cache without going through the SPL executor) ──────
def cache_get(concept: str, rubric_version: str = "v1") -> str:
    """Returns cached content on a hit, or the sentinel string "miss"."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().get(
            concept=concept, params={}, rubric_version=rubric_version, dep_hashes={},
        )
        return entry.content if entry is not None else "miss"
    except Exception:
        return "miss"

def cache_put(concept: str, content: str, rubric_version: str = "v1") -> str:
    """Stores generated+verified content (write-once, immutable); returns the cache key."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().put(
            concept=concept, content=content, provenance="machine_generated",
            params={}, rubric_version=rubric_version, dep_hashes={},
        )
        return entry.key
    except Exception as exc:
        return f"cache_put error: {exc}"

# ── Domain verifier helpers — one well-known helper per CALL target this
# .spl source can dispatch to (declared via DomainConfig.verifiers; see
# python/linalg's preset). Each returns 'pass' or a 'fail: ...' description an
# EVALUATE @check WHEN contains("fail") branch can act on. ───────────────────
# ── Math verifier helpers ─────────────────────────────────────────────────────
def _verify_math(section: str) -> str:
    """Returns 'pass' or a description of the failure."""
    try:
        import sympy  # noqa: F401 — presence check
        return "pass"   # TODO: parse worked examples from section and recompute
    except Exception as exc:
        return f"fail: {exc}"

def _shape_check(section: str) -> str:
    """Check matrix dimension compatibility. Returns 'pass' or error."""
    return "pass"  # TODO: parse matrix expressions and check shapes

# ── Timing helpers ────────────────────────────────────────────────────────────
# Bare-name wrappers around time.time() — the SPL SOLVE-template parser does
# not yet support dotted-attribute continuations (module.attr(...)), so the
# .spl source calls these instead of `time.time()` directly. Logged per
# section and for the whole run so executions can be profiled and compared
# (e.g. across distributed workers).
def _now() -> float:
    return time.time()

def _elapsed(start: float) -> float:
    return time.time() - start

# ── Workflow parameters (override before running or use papermill) ────────────
target = _spl_config('TARGET', 'spectral_theorem')  # TEXT
style = _spl_config('STYLE', 'textbook')  # TEXT
primitive_budget = int(_spl_config('PRIMITIVE_BUDGET', str(2)))  # INT
payoff_weight = float(_spl_config('PAYOFF_WEIGHT', str(1.5)))  # FLOAT
textbook = ""

# Pre-build the linalg_graph concept graph (reused by all SOLVE/ASSERT cells)
graph = dg.build()
primitives = dg.both_radical_primitives()
print(f"linalg_graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

linalg_graph loaded: 37 nodes, 51 edges


In [2]:
# ── Prompt templates (mirrors CREATE FUNCTION blocks in .spl) ────────────

WRITE_SECTION_PROMPT = """\
You are an expert linear algebra author.

{style_guide}

Write a self-contained section for the concept: {concept}
Surrounding context: {context}

Follow the structure and length specified in the STYLE GUIDE above exactly.
Use LaTeX for all mathematics.
Output only the section text. No preamble, no metadata.
"""

REFINE_SECTION_PROMPT = """\
You are an expert linear algebra author. Revise the section below based on
the feedback provided. Keep all correct content; fix only what is flagged.
Maintain the style described below throughout the revision.

{style_guide}

SECTION:
{section}

FEEDBACK:
{feedback}

Output the revised section only. No preamble. No commentary.
"""

WRITE_PAYOFF_PROMPT = """\
You are an expert author writing the capstone of a linear algebra micro-textbook.

{style_guide}

The textbook culminates in the concept: {target}
Applications this concept unlocks: {applications}

Write a "Payoff" section that:
1. Explains what {target} achieves and why it is the natural endpoint.
2. Shows how it connects to each listed application domain.
3. Ends with an invitation to explore one application in depth.

Follow the length and structure in the STYLE GUIDE above.
Use LaTeX for mathematics. Output only the section. No preamble.
"""

In [3]:
# SPL: SOLVE @t_workflow_start FLOAT := _now()
t_workflow_start = _now()
print(f'@t_workflow_start =', t_workflow_start)

@t_workflow_start = 1780891464.3712788


In [4]:
# SPL: SOLVE @style_guide TEXT := style_instruction(@style)
style_guide = style_instruction(style)
print(f'@style_guide =', style_guide)

@style_guide = STYLE GUIDE — University textbook
Tone      : precise and formal
Depth     : full definition, proof sketch, concrete worked example
Audience  : first-year university student with calculus background
Length    : 300–400 words per section
Structure : Definition → Worked example → Key theorem → Lab cell (SymPy)


In [5]:
# SPL: LOGGING f'Style: {@style}' LEVEL INFO
print('[INFO]', f"Style: {style}")

[INFO] Style: textbook


In [6]:
# SPL: ASSERT acyclic(@graph)
_assert_result = bool(acyclic(graph))
if not _assert_result:
    raise AssertionError(f"ASSERT failed: 'acyclic(graph)'")
else:
    print(f"✓ ASSERT acyclic(graph)")

✓ ASSERT acyclic(graph)


In [7]:
# SPL: ASSERT reducible(@graph, @primitives)
_assert_result = bool(reducible(graph, primitives))
if not _assert_result:
    raise AssertionError(f"ASSERT failed: 'reducible(graph, primitives)'")
else:
    print(f"✓ ASSERT reducible(graph, primitives)")

✓ ASSERT reducible(graph, primitives)


In [8]:
# SPL: SOLVE @needed SET := ancestors(@graph, @target)
needed = ancestors(graph, target)
print(f'@needed =', needed)

@needed = {'scalar_multiplication', 'linear_combination', 'vector_addition'}


In [9]:
# SPL: SOLVE @order LIST := productivity_order(restrict(@graph, @needed), weight=@payoff_weight)
order = productivity_order(restrict(graph, needed), weight=payoff_weight)
print(f'@order =', order)

@order = ['scalar_multiplication', 'vector_addition', 'linear_combination']


In [10]:
# SPL: SOLVE @order_len INT := len(@order)
order_len = len(order)
print(f'@order_len =', order_len)

@order_len = 3


In [11]:
# SPL: LOGGING f'Teaching {@order_len} concepts toward {@target}' LEVEL INFO
print('[INFO]', f"Teaching {order_len} concepts toward {target}")

[INFO] Teaching 3 concepts toward linear_independence


In [12]:
# SPL: @textbook := ''
textbook = ''
print(f'@textbook =', textbook)

@textbook = 


In [13]:
# SPL: @spl__i := 0
spl__i = 0
print(f'@spl__i =', spl__i)

@spl__i = 0


In [14]:
# SPL: WHILE @spl__i < @order_len DO
_while_iter = 0
while (spl__i < order_len) and _while_iter < (order_len + 1):
    _while_iter += 1
    # SPL: SOLVE @concept TEXT := @order[@_i]
    concept = order[spl__i]
    print(f'@concept =', concept)
    # SPL: SOLVE @t_section_start FLOAT := _now()
    t_section_start = _now()
    print(f'@t_section_start =', t_section_start)
    # SPL: LOGGING f'Section {@_i}: {@concept}' LEVEL INFO
    print('[INFO]', f"Section {spl__i}: {concept}")
    # SPL: CALL cache_get(@concept) INTO @section
    section = cache_get(concept)
    print(f'@section:', section)
    # SPL: EVALUATE @section
    if section == 'miss':
        # SPL: LOGGING f'cache MISS for {@concept} — generating with the LLM' LEVEL INFO
        print('[INFO]', f"cache MISS for {concept} — generating with the LLM")
        # SPL: GENERATE write_section(@concept, @concept, @style_guide) INTO @section
        _prompt = WRITE_SECTION_PROMPT.format(concept=concept, context=concept, style_guide=style_guide)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
        # SPL: ASSERT new_primitives(@section)<=@primitive_budget
        _assert_result = bool(new_primitives(section)<=primitive_budget)
        if not _assert_result:
            print(f"ASSERT failed: 'new_primitives(section)<=primitive_budget' → executing OTHERWISE")
            # SPL: GENERATE refine_section(@section, 'introduce at most one new primitive per section', @style_guide) INTO @section
            _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback='introduce at most one new primitive per section', style_guide=style_guide)
            section = _llm_call(_prompt)
            print(f'@section:', section[:200] if isinstance(section, str) else section)
        else:
            print(f"✓ ASSERT new_primitives(section)<=primitive_budget")
        # SPL: CALL verify_math(@section) INTO @check
        check = _verify_math(section)
        print(f'@check:', check)
        # SPL: EVALUATE @check
        if "fail" in str(check).lower():
            # SPL: GENERATE refine_section(@section, @check, @style_guide) INTO @section
            _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=check, style_guide=style_guide)
            section = _llm_call(_prompt)
            print(f'@section:', section[:200] if isinstance(section, str) else section)
        else:
            # SPL: LOGGING 'Math verified' LEVEL INFO
            print('[INFO]', 'Math verified')
        # SPL: CALL shape_check(@section) INTO @shapes
        shapes = _shape_check(section)
        print(f'@shapes:', shapes)
        # SPL: EVALUATE @shapes
        if "fail" in str(shapes).lower():
            # SPL: GENERATE refine_section(@section, @shapes, @style_guide) INTO @section
            _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=shapes, style_guide=style_guide)
            section = _llm_call(_prompt)
            print(f'@section:', section[:200] if isinstance(section, str) else section)
        else:
            # SPL: LOGGING 'Shapes OK' LEVEL INFO
            print('[INFO]', 'Shapes OK')
        # SPL: CALL cache_put(@concept, @section) INTO @cache_key
        cache_key = cache_put(concept, section)
        print(f'@cache_key:', cache_key)
        # SPL: LOGGING f'Stored verified section — cache key {@cache_key}' LEVEL INFO
        print('[INFO]', f"Stored verified section — cache key {cache_key}")
    else:
        # SPL: LOGGING f'cache HIT for {@concept} — reusing verified section (0 LLM calls)' LEVEL INFO
        print('[INFO]', f"cache HIT for {concept} — reusing verified section (0 LLM calls)")
    # SPL: SOLVE @section_elapsed FLOAT := _elapsed(@t_section_start)
    section_elapsed = _elapsed(t_section_start)
    print(f'@section_elapsed =', section_elapsed)
    # SPL: LOGGING f'[timing] {@concept}: {@section_elapsed:.1f}s' LEVEL INFO
    print('[INFO]', f"[timing] {concept}: {section_elapsed:.1f}s")
    # SPL: @textbook := @textbook + '\n\n---\n\n' + @section
    textbook = textbook + '\n\n---\n\n' + section
    print(f'@textbook =', textbook)
    # SPL: @spl__i := @spl__i + 1
    spl__i = spl__i + 1
    print(f'@spl__i =', spl__i)
# END WHILE  (ran {_while_iter} iteration(s))

@concept = scalar_multiplication
@t_section_start = 1780891464.4731727
[INFO] Section 0: scalar_multiplication
@section: miss
[INFO] cache MISS for scalar_multiplication — generating with the LLM
@section: ### Scalar Multiplication

Scalar multiplication is a fundamental operation in linear algebra that extends vector addition to operations with scalars. It involves multiplying every component of a vect
✓ ASSERT new_primitives(section)<=primitive_budget
@check: pass
[INFO] Math verified
@shapes: pass
[INFO] Shapes OK
@cache_key: spl3:content:bd759fec5df5dba2e03453f60ac790bae5efefb0d3b7fb25b6ba3241296156a5
[INFO] Stored verified section — cache key spl3:content:bd759fec5df5dba2e03453f60ac790bae5efefb0d3b7fb25b6ba3241296156a5
@section_elapsed = 12.670706987380981
[INFO] [timing] scalar_multiplication: 12.7s
@textbook = 

---

### Scalar Multiplication

Scalar multiplication is a fundamental operation in linear algebra that extends vector addition to operations with scalars. It involves m

In [15]:
# SPL: SOLVE @apps LIST := applications_of(@graph, @target)
apps = applications_of(graph, target)
print(f'@apps =', apps)

@apps = []


In [16]:
# SPL: SOLVE @t_payoff_start FLOAT := _now()
t_payoff_start = _now()
print(f'@t_payoff_start =', t_payoff_start)

@t_payoff_start = 1780891499.513061


In [17]:
# SPL: GENERATE write_payoff(@target, @apps, @style_guide) INTO @capstone
_prompt = WRITE_PAYOFF_PROMPT.format(target=target, applications=apps, style_guide=style_guide)
capstone = _llm_call(_prompt)
print(f'@capstone:', capstone[:200] if isinstance(capstone, str) else capstone)

@capstone: ## Section 6.1: Vectors in Euclidean Space

This section introduces vectors within a Euclidean space, formally defining them as objects possessing both magnitude and direction. Our primary focus will 


In [18]:
# SPL: SOLVE @payoff_elapsed FLOAT := _elapsed(@t_payoff_start)
payoff_elapsed = _elapsed(t_payoff_start)
print(f'@payoff_elapsed =', payoff_elapsed)

@payoff_elapsed = 19.553279876708984


In [19]:
# SPL: LOGGING f'[timing] payoff: {@payoff_elapsed:.1f}s' LEVEL INFO
print('[INFO]', f"[timing] payoff: {payoff_elapsed:.1f}s")

[INFO] [timing] payoff: 19.6s


In [20]:
# SPL: @textbook := @textbook + '\n\n---\n\n## Payoff\n\n' + @capstone
textbook = textbook + '\n\n---\n\n## Payoff\n\n' + capstone
print(f'@textbook =', textbook)

@textbook = 

---

### Scalar Multiplication

Scalar multiplication is a fundamental operation in linear algebra that extends vector addition to operations with scalars. It involves multiplying every component of a vector by a single scalar value. This process fundamentally alters the magnitude (length) of the vector while preserving its direction (assuming the scalar is positive). Let's explore this concept in detail.

**Definition:**
Given a vector **v** = \begin{pmatrix} v_1 \\ v_2 \\ \vdots \\ v_n \end{pmatrix} ∈ ℝ<sup>n</sup> and a scalar *k* ∈ ℝ, the scalar multiplication is defined as:

  **k** **v** = \begin{pmatrix} k v_1 \\ k v_2 \\ \vdots \\ k v_n \end{pmatrix}

This operation effectively scales each component of the vector by the factor *k*. If *k* > 0, the resulting vector has the same direction as **v**. If *k* < 0, the resulting vector points in the opposite direction as **v**.  If *k* = 0, the result is the zero vector.

**Worked Example:**
Let’s consider a two-dimensio

In [21]:
# SPL: SOLVE @total_elapsed FLOAT := _elapsed(@t_workflow_start)
total_elapsed = _elapsed(t_workflow_start)
print(f'@total_elapsed =', total_elapsed)

@total_elapsed = 54.71443319320679


In [22]:
# SPL: LOGGING f'[timing] workflow total: {@total_elapsed:.1f}s for {@order_len} sections + payoff' LEVEL INFO
print('[INFO]', f"[timing] workflow total: {total_elapsed:.1f}s for {order_len} sections + payoff")

[INFO] [timing] workflow total: 54.7s for 3 sections + payoff


In [23]:
# SPL: LOGGING f'Textbook complete — style: {@style}' LEVEL INFO
print('[INFO]', f"Textbook complete — style: {style}")

[INFO] Textbook complete — style: textbook


In [24]:
# SPL: COMMIT @textbook
_output = textbook
_result_path = Path("build_micro_textbook_output.md")
_result_path.write_text(_output, encoding='utf-8')
print(f"Committed to {_result_path}")

Committed to build_micro_textbook_output.md
